In [1]:
!git clone https://github.com/thepoojashinde/medical-image-denoising-v2.git
%cd medical-image-denoising-v2

Cloning into 'medical-image-denoising-v2'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 29 (delta 8), reused 24 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 5.11 MiB | 21.97 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/kaggle/working/medical-image-denoising-v2


**INSTALLING DEPENDENCIES**

In [2]:
import os
os.chdir('/kaggle/working/medical-image-denoising-v2')
from unet_model_4ch import unet_model_4ch
print("Ready!")

2026-06-25 07:15:37.342260: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782371737.542718      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782371737.598712      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782371738.074049      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782371738.074101      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782371738.074104      58 computation_placer.cc:177] computation placer alr

4ch U-Net defined!
Ready!


In [3]:
!pip install pydicom PyWavelets scikit-image -q

**writing main.py**

In [4]:
print("Step 1: Walking dataset...")
dicom_paths = []
for root, _, files in os.walk('/kaggle/input/datasets/thepoojashinde/lidc-idri-denoising-subset/dataset/manifest-1600709154662/LIDC-IDRI/'):
    for f in files:
        if f.endswith('.dcm'):
            dicom_paths.append(os.path.join(root, f))
print(f"Found {len(dicom_paths)} DICOM files.")

Step 1: Walking dataset...
Found 3629 DICOM files.


In [5]:
!pip install pydicom PyWavelets scikit-image bm3d -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.0/862.0 kB 9.1 MB/s eta 0:00:00:00:010:01


In [6]:
# ==============================================================
# FULL PIPELINE: Retrain DnCNN, ResUNet, Proposed (4ch DWT+UNet)
# + Unified evaluation (MSE, PSNR, SSIM, FSIM) -- no hardcoding
# + BM3D using Option B (sigma = injected noise std)
# + Saves: per-image per-metric tables (sir's Table I-VIII style),
#          Fig.2-style visual grid, training curves, raw backups
# ==============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import pydicom
import pywt
import bm3d
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization,
                                      Activation, Add, MaxPooling2D,
                                      UpSampling2D, Concatenate)
from tensorflow.keras.models import Model

np.random.seed(42)

OUTPUT_DIR = "/kaggle/working/paper_results_gaussian_only"
BACKUP_DIR = "/kaggle/working/paper_results_gaussian_only_backup"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(BACKUP_DIR, exist_ok=True)


# ------------------------------------------------------------
# 1. HELPER FUNCTIONS
# ------------------------------------------------------------

def load_dicom(path, size=(128, 128)):
    dcm = pydicom.dcmread(path)
    img = dcm.pixel_array.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return cv2.resize(img, size)

def add_gaussian_noise(img, mean=0, std=0.05):
    return np.clip(img + np.random.normal(mean, std, img.shape), 0, 1)

def apply_dwt(img_2d, wavelet='haar'):
    LL, (LH, HL, HH) = pywt.dwt2(img_2d, wavelet)
    return LL, (LH, HL, HH)

def apply_idwt(LL, high_freq_bands, wavelet='haar'):
    return pywt.idwt2((LL, high_freq_bands), wavelet)

def fsim(img1, img2):
    """Feature Similarity Index."""
    def gradient_magnitude(img):
        img_uint8 = (img * 255).astype(np.uint8)
        gx = cv2.Sobel(img_uint8, cv2.CV_64F, 1, 0, ksize=3)
        gy = cv2.Sobel(img_uint8, cv2.CV_64F, 0, 1, ksize=3)
        return np.sqrt(gx**2 + gy**2)
    T1, T2 = 0.85, 160.0
    PC1 = gradient_magnitude(img1)
    PC2 = gradient_magnitude(img2)
    PCm = np.maximum(PC1, PC2)
    S_PC = (2 * PC1 * PC2 + T1) / (PC1**2 + PC2**2 + T1)
    S_G  = (2 * PC1 * PC2 + T2) / (PC1**2 + PC2**2 + T2)
    return np.sum(S_PC * S_G * PCm) / (np.sum(PCm) + 1e-8)

def compute_all_metrics(clean, denoised):
    """Single source of truth -- all 4 metrics computed together, every time."""
    mse_val  = mean_squared_error(clean.flatten(), denoised.flatten())
    psnr_val = psnr(clean, denoised, data_range=1.0)
    ssim_val = ssim(clean, denoised, data_range=1.0)
    fsim_val = fsim(clean, denoised)
    return mse_val, psnr_val, ssim_val, fsim_val


# ------------------------------------------------------------
# 2. DATASET BUILDING (4-channel DWT subbands, shared by all 3 trained models)
# ------------------------------------------------------------

def build_dwt_dataset_4ch(dicom_paths, wavelet='haar'):

    X = []
    Y = []
    noise_levels = []

    for path in dicom_paths:
        try:
            # Load clean CT image
            clean = load_dicom(path)

            # Random Gaussian noise level
            std = np.random.uniform(0.01, 0.09)
            noise_levels.append(std)

            # Generate noisy image
            noisy = add_gaussian_noise(clean, std=std)

            # DWT decomposition
            LL_c, (LH_c, HL_c, HH_c) = apply_dwt(clean, wavelet)
            LL_n, (LH_n, HL_n, HH_n) = apply_dwt(noisy, wavelet)

            # Stack into 4-channel tensors
            clean_coeffs = np.stack([LL_c, LH_c, HL_c, HH_c], axis=-1)
            noisy_coeffs = np.stack([LL_n, LH_n, HL_n, HH_n], axis=-1)

            X.append(noisy_coeffs)
            Y.append(clean_coeffs)

        except Exception as e:
            print(f"Error processing {path}: {e}")

    X = np.array(X, dtype=np.float32)
    Y = np.array(Y, dtype=np.float32)

    print(f"Dataset created successfully.")
    print(f"Input shape : {X.shape}")
    print(f"Target shape: {Y.shape}")

    return X, Y, np.array(noise_levels, dtype=np.float32)


# ------------------------------------------------------------
# 3. MODEL ARCHITECTURES
# ------------------------------------------------------------

def dncnn_model(input_shape=(64, 64, 4), depth=8, filters=64):
    inputs = Input(shape=input_shape)
    x = Conv2D(filters, 3, padding='same', activation='relu')(inputs)
    for _ in range(depth - 2):
        x = Conv2D(filters, 3, padding='same', use_bias=False)(x)
        x = BatchNormalization()(x)
        x = Activation('relu')(x)
    x = Conv2D(4, 3, padding='same')(x)
    outputs = Add()([inputs, x])
    return Model(inputs, outputs, name="DnCNN")

def residual_block(x, filters):
    shortcut = Conv2D(filters, 1, padding='same')(x)
    x = Conv2D(filters, 3, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(filters, 3, padding='same')(x)
    x = BatchNormalization()(x)
    x = Add()([x, shortcut])
    return Activation('relu')(x)



# unet_model_4ch is imported from your repo (unet_model_4ch.py) -- kept as-is
from unet_model_4ch import unet_model_4ch


# ------------------------------------------------------------
# 4. TRAINING (all 3 models, same data, same EarlyStopping policy)
# ------------------------------------------------------------

def train_model(model, X_train, Y_train, save_path, model_name):
    model.compile(optimizer=Adam(1e-3), loss='mse', metrics=['mae'])
    checkpoint = ModelCheckpoint(save_path, monitor='val_loss',
                                  save_best_only=True, verbose=0)
    early_stop = EarlyStopping(monitor='val_loss', patience=5,
                                 restore_best_weights=True)
    history = model.fit(
        X_train, Y_train,
        epochs=50, batch_size=16, validation_split=0.1,
        callbacks=[checkpoint, early_stop], verbose=1
    )
    model.save(save_path)
    print(f"[done] {model_name} trained and saved -> {save_path}")
    return history


# ------------------------------------------------------------
# 5. METHOD WRAPPERS -- every method takes a 2D noisy image, returns 2D denoised
#    All deep models go through the SAME DWT -> predict -> IDWT pipeline.
# ------------------------------------------------------------

def denoise_with_bm3d(noisy_2d, sigma_psd):
    """Option B: sigma_psd = the injected noise std itself (known, since noise is synthetic)."""
    denoised = bm3d.bm3d(noisy_2d, sigma_psd=sigma_psd)
    return np.clip(denoised, 0, 1)

def denoise_dwt_model(noisy_2d, model):
    LL_n, (LH_n, HL_n, HH_n) = apply_dwt(noisy_2d)
    inp = np.stack([LL_n, LH_n, HL_n, HH_n], axis=-1)[np.newaxis]
    out = model.predict(inp, verbose=0)[0]
    reconstructed = apply_idwt(out[:, :, 0], (out[:, :, 1], out[:, :, 2], out[:, :, 3]))
    return np.clip(reconstructed, 0, 1)


# ------------------------------------------------------------
# 6. MASTER EVALUATION LOOP -- same noisy input reused across all methods
# ------------------------------------------------------------

def run_full_evaluation(external_paths, models, std_values=None):
    """
    models : dict {"DnCNN": model, "ResUNet": model, "Proposed": model}
    For each image x each noise level: ONE noisy image generated,
    reused identically across BM3D, DnCNN, ResUNet, Proposed.
    Saves per-image per-metric CSVs in sir's Table I-VIII format.
    """
    if std_values is None:
        std_values = np.linspace(0.01, 0.09, 9)  # matches sir's sigma^2 range

    methods = {
        "BM3D":    lambda noisy, std: denoise_with_bm3d(noisy, sigma_psd=std),
        "DnCNN":   lambda noisy, std: denoise_dwt_model(noisy, models["DnCNN"]),
        "ResUNet": lambda noisy, std: denoise_dwt_model(noisy, models["ResUNet"]),
        "Proposed": lambda noisy, std: denoise_dwt_model(noisy, models["Proposed"]),
    }

    all_image_tables = {}
    raw_backup = {}  # keep clean/noisy/denoised arrays for safety

    for idx, path in enumerate(external_paths):
        image_id = f"Image{idx+1}"
        clean = load_dicom(path)
        raw_backup[image_id] = {"clean": clean, "noisy": {}, "denoised": {m: {} for m in methods}}

        rows = []
        for std in std_values:
            noisy = add_gaussian_noise(clean, std=std)  # SAME noisy image for every method
            raw_backup[image_id]["noisy"][round(std, 3)] = noisy
            row = {"sigma": round(std, 3)}

            for method_name, method_fn in methods.items():
                denoised = method_fn(noisy, std)
                raw_backup[image_id]["denoised"][method_name][round(std, 3)] = denoised
                mse_val, psnr_val, ssim_val, fsim_val = compute_all_metrics(clean, denoised)
                row[f"MSE_{method_name}"]  = round(mse_val, 6)
                row[f"PSNR_{method_name}"] = round(psnr_val, 2)
                row[f"SSIM_{method_name}"] = round(ssim_val, 6)
                row[f"FSIM_{method_name}"] = round(fsim_val, 6)

            rows.append(row)

        df = pd.DataFrame(rows)
        df.to_csv(os.path.join(OUTPUT_DIR, f"{image_id}_full_table.csv"), index=False)
        all_image_tables[image_id] = df

        # Per-metric tables -- matches sir's Table I/II/III/IV layout exactly
        for metric in ["MSE", "PSNR", "SSIM", "FSIM"]:
            cols = ["sigma"] + [f"{metric}_{m}" for m in methods.keys()]
            metric_df = df[cols].copy()
            metric_df.columns = ["sigma"] + list(methods.keys())
            metric_df.to_csv(os.path.join(OUTPUT_DIR, f"{image_id}_{metric}_table.csv"), index=False)

        print(f"[done] {image_id} -> tables saved")

    # Backup raw arrays (clean/noisy/denoised) as .npz so nothing is lost
    np.savez_compressed(os.path.join(BACKUP_DIR, "raw_arrays_backup.npz"),
                         data=raw_backup, allow_pickle=True)
    print(f"[done] raw arrays backed up -> {BACKUP_DIR}/raw_arrays_backup.npz")

    return all_image_tables


# ------------------------------------------------------------
# 7. FIG.2-STYLE VISUAL GRID (saved exactly, per your request)
# ------------------------------------------------------------

def generate_comparison_figure(external_paths, models, std=0.07,
                                output_path=None):
    if output_path is None:
        output_path = os.path.join(OUTPUT_DIR, "fig2_comparison.png")

    methods = {
        "BM3D":    lambda noisy: denoise_with_bm3d(noisy, sigma_psd=std),
        "DnCNN":   lambda noisy: denoise_dwt_model(noisy, models["DnCNN"]),
        "ResUNet": lambda noisy: denoise_dwt_model(noisy, models["ResUNet"]),
        "Proposed": lambda noisy: denoise_dwt_model(noisy, models["Proposed"]),
    }

    n_images = len(external_paths)
    n_cols = len(methods) + 1  # +1 for "Noisy" reference column
    fig, axes = plt.subplots(n_images, n_cols, figsize=(3 * n_cols, 3 * n_images))

    for row, path in enumerate(external_paths):
        clean = load_dicom(path)
        noisy = add_gaussian_noise(clean, std=std)

        axes[row, 0].imshow(noisy, cmap="gray")
        axes[row, 0].set_title("Noisy" if row == 0 else "")
        axes[row, 0].axis("off")

        for col, (method_name, method_fn) in enumerate(methods.items(), start=1):
            denoised = method_fn(noisy)
            axes[row, col].imshow(denoised, cmap="gray")
            axes[row, col].set_title(method_name if row == 0 else "")
            axes[row, col].axis("off")

    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()
    print(f"[done] comparison figure saved -> {output_path}")


# ------------------------------------------------------------
# 8. TRAINING CURVES (saved for all 3 trained models)
# ------------------------------------------------------------

def save_training_curve(history, model_name):
    plt.figure(figsize=(8, 5))
    plt.plot(history.history["loss"], label="Training Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Training Curve of {model_name}")
    plt.legend()
    plt.grid(True)
    path = os.path.join(OUTPUT_DIR, f"{model_name.lower().replace(' ', '_')}_training_curve.png")
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"[done] training curve saved -> {path}")

print("THIS CELL RUNS SUCCESSFULLY!")
# ------------------------------------------------------------
# 9. FULL RUN (this is what you execute in Kaggle)
# ------------------------------------------------------------


THIS CELL RUNS SUCCESSFULLY!


In [7]:
from tensorflow.keras.layers import (
    Input, Conv2D, BatchNormalization, Activation,
    Add, UpSampling2D, Concatenate
)
from tensorflow.keras.models import Model


# ---------------------------------------------------
# Residual Block (Zhang et al.)
# ---------------------------------------------------
def res_block(x, filters, strides=((1,1),(1,1))):

    shortcut = Conv2D(
        filters[1],
        kernel_size=1,
        strides=strides[0],
        padding="same"
    )(x)
    shortcut = BatchNormalization()(shortcut)

    out = BatchNormalization()(x)
    out = Activation("relu")(out)

    out = Conv2D(
        filters[0],
        3,
        strides=strides[0],
        padding="same"
    )(out)

    out = BatchNormalization()(out)
    out = Activation("relu")(out)

    out = Conv2D(
        filters[1],
        3,
        strides=strides[1],
        padding="same"
    )(out)

    out = Add()([shortcut, out])

    return out


# ---------------------------------------------------
# ResUNet
# ---------------------------------------------------
def resunet_model(input_shape=(64,64,4)):

    inputs = Input(shape=input_shape)

    # ---------------- Encoder ----------------

    e1 = Conv2D(64,3,padding="same")(inputs)
    e1 = BatchNormalization()(e1)
    e1 = Activation("relu")(e1)
    e1 = Conv2D(64,3,padding="same")(e1)

    sc = Conv2D(64,1,padding="same")(inputs)
    sc = BatchNormalization()(sc)

    e1 = Add()([e1,sc])

    e2 = res_block(
        e1,
        [128,128],
        strides=((2,2),(1,1))
    )

    e3 = res_block(
        e2,
        [256,256],
        strides=((2,2),(1,1))
    )

    # ---------------- Bottleneck ----------------
    # No extra downsampling

    b = res_block(
        e3,
        [512,512],
        strides=((1,1),(1,1))
    )

    # ---------------- Decoder ----------------

    d1 = UpSampling2D((2,2))(b)
    d1 = Concatenate()([d1,e2])
    d1 = res_block(d1,[256,256])

    d2 = UpSampling2D((2,2))(d1)
    d2 = Concatenate()([d2,e1])
    d2 = res_block(d2,[128,128])

    outputs = Conv2D(
        4,
        1,
        activation="linear",
        padding="same"
    )(d2)

    return Model(inputs, outputs, name="ResUNet")

In [8]:
# ==============================================================
# FAIR FINAL EVALUATION -- all 4 methods on the SAME full test set
# No sample-size mismatch (BM3D now runs on full X_test too)
# Saves everything into /kaggle/working/researchPaperFinals
# ==============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import bm3d
from sklearn.metrics import mean_squared_error
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

FINAL_DIR = OUTPUT_DIR
os.makedirs(FINAL_DIR, exist_ok=True)
print("Saving everything to:", FINAL_DIR)


def compute_metrics(clean, denoised):
    mse_val  = mean_squared_error(clean.flatten(), denoised.flatten())
    psnr_val = psnr(clean, denoised, data_range=1.0)
    ssim_val = ssim(clean, denoised, data_range=1.0)
    fsim_val = fsim(clean, denoised)
    return mse_val, psnr_val, ssim_val, fsim_val


# ------------------------------------------------------------
# 1. FULL TEST-SET EVALUATION -- same sample count for ALL methods
# ------------------------------------------------------------

def evaluate_full_testset(
    X_test,
    Y_test,
    noise_test,
    models,
    n_samples=None
):
    """
    models : dict {"DnCNN": model_dncnn, "ResUNet": model_resunet, "Proposed": model_proposed}
    n_samples : if None, uses the FULL test set for every method (BM3D included).
                Set a number only if you want a smaller run for a quick check.
    """
    n_total = len(X_test)
    n_samples = n_total if n_samples is None else min(n_samples, n_total)
    indices = np.arange(n_samples)  # same fixed indices for every method -> fair comparison

    results = []

    # --- Deep learning models ---
    for method_name, model in models.items():
        print(f"\nEvaluating {method_name} on {n_samples} samples...")
        mse_scores, psnr_scores, ssim_scores, fsim_scores = [], [], [], []

        for i in indices:
            pred = model.predict(X_test[i:i+1], verbose=0)[0]
            pred_img  = apply_idwt(pred[:,:,0],  (pred[:,:,1],  pred[:,:,2],  pred[:,:,3]))
            clean_img = apply_idwt(Y_test[i,:,:,0], (Y_test[i,:,:,1], Y_test[i,:,:,2], Y_test[i,:,:,3]))
            pred_img, clean_img = np.clip(pred_img, 0, 1), np.clip(clean_img, 0, 1)

            mse_val, psnr_val, ssim_val, fsim_val = compute_metrics(clean_img, pred_img)
            mse_scores.append(mse_val); psnr_scores.append(psnr_val)
            ssim_scores.append(ssim_val); fsim_scores.append(fsim_val)

        results.append({
            "Method": method_name,
            "MSE": np.mean(mse_scores), "PSNR": np.mean(psnr_scores),
            "SSIM": np.mean(ssim_scores), "FSIM": np.mean(fsim_scores),
            "N_samples": n_samples
        })

    # --- BM3D -- SAME n_samples and SAME indices as deep models, no shortcut ---
    print(f"\nEvaluating BM3D on {n_samples} samples (this will take a while)...")
    mse_scores, psnr_scores, ssim_scores, fsim_scores = [], [], [], []

    for count, i in enumerate(indices):
        if count % 50 == 0:
            print(f"  BM3D progress: {count}/{n_samples}")

        noisy_img = apply_idwt(X_test[i,:,:,0], (X_test[i,:,:,1], X_test[i,:,:,2], X_test[i,:,:,3]))
        clean_img = apply_idwt(Y_test[i,:,:,0], (Y_test[i,:,:,1], Y_test[i,:,:,2], Y_test[i,:,:,3]))
        noisy_img, clean_img = np.clip(noisy_img, 0, 1), np.clip(clean_img, 0, 1)

        sigma = float(noise_test[i])

        denoised = bm3d.bm3d(
            noisy_img,
            sigma_psd=sigma
        )

        mse_val, psnr_val, ssim_val, fsim_val = compute_metrics(clean_img, denoised)
        mse_scores.append(mse_val); psnr_scores.append(psnr_val)
        ssim_scores.append(ssim_val); fsim_scores.append(fsim_val)

    results.append({
        "Method": "BM3D",
        "MSE": np.mean(mse_scores), "PSNR": np.mean(psnr_scores),
        "SSIM": np.mean(ssim_scores), "FSIM": np.mean(fsim_scores),
        "N_samples": n_samples
    })

    comparison = pd.DataFrame(results)
    comparison.to_csv(os.path.join(FINAL_DIR, "final_testset_comparison.csv"), index=False)
    print("\n" + "="*60)
    print(f"FINAL TEST SET RESULTS (n={n_samples}, same for all methods)")
    print("="*60)
    print(comparison.to_string(index=False))
    return comparison


# ------------------------------------------------------------
# 2. PER-EXTERNAL-IMAGE TABLES (sir's Table I-VIII style) + Fig.2 grid
#    Reuses your existing helper functions (load_dicom, add_gaussian_noise,
#    apply_dwt, apply_idwt, denoise_dwt_model, denoise_with_bm3d, fsim)
#    -- these must already be defined earlier in your notebook.
# ------------------------------------------------------------

def run_external_image_tables(external_paths, models, std_values=None):
    if std_values is None:
        std_values = np.linspace(0.01, 0.09, 9)

    methods = {
        "BM3D":     lambda noisy, std: denoise_with_bm3d(noisy, sigma_psd=std),
        "DnCNN":    lambda noisy, std: denoise_dwt_model(noisy, models["DnCNN"]),
        "ResUNet":  lambda noisy, std: denoise_dwt_model(noisy, models["ResUNet"]),
        "Proposed": lambda noisy, std: denoise_dwt_model(noisy, models["Proposed"]),
    }

    all_tables = {}
    for idx, path in enumerate(external_paths):
        image_id = f"Image{idx+1}"
        clean = load_dicom(path)
        rows = []
        for std in std_values:
            noisy = add_gaussian_noise(clean, std=std)
            row = {"sigma": round(std, 3)}
            for method_name, method_fn in methods.items():
                denoised = method_fn(noisy, std)
                mse_val, psnr_val, ssim_val, fsim_val = compute_metrics(clean, denoised)
                row[f"MSE_{method_name}"]  = round(mse_val, 6)
                row[f"PSNR_{method_name}"] = round(psnr_val, 2)
                row[f"SSIM_{method_name}"] = round(ssim_val, 6)
                row[f"FSIM_{method_name}"] = round(fsim_val, 6)
            rows.append(row)

        df = pd.DataFrame(rows)
        df.to_csv(os.path.join(FINAL_DIR, f"{image_id}_full_table.csv"), index=False)
        all_tables[image_id] = df

        for metric in ["MSE", "PSNR", "SSIM", "FSIM"]:
            cols = ["sigma"] + [f"{metric}_{m}" for m in methods.keys()]
            metric_df = df[cols].copy()
            metric_df.columns = ["sigma"] + list(methods.keys())
            metric_df.to_csv(os.path.join(FINAL_DIR, f"{image_id}_{metric}_table.csv"), index=False)
        print(f"[done] {image_id} tables saved")

    return all_tables


def generate_comparison_figure(external_paths, models, std=0.07):
    output_path = os.path.join(FINAL_DIR, "fig2_comparison.png")
    methods = {
        "BM3D":     lambda noisy: denoise_with_bm3d(noisy, sigma_psd=std),
        "DnCNN":    lambda noisy: denoise_dwt_model(noisy, models["DnCNN"]),
        "ResUNet":  lambda noisy: denoise_dwt_model(noisy, models["ResUNet"]),
        "Proposed": lambda noisy: denoise_dwt_model(noisy, models["Proposed"]),
    }
    n_images = len(external_paths)
    n_cols = len(methods) + 1
    fig, axes = plt.subplots(n_images, n_cols, figsize=(3*n_cols, 3*n_images))

    for row, path in enumerate(external_paths):
        clean = load_dicom(path)
        noisy = add_gaussian_noise(clean, std=std)
        axes[row, 0].imshow(noisy, cmap="gray")
        axes[row, 0].set_title("Noisy" if row == 0 else "")
        axes[row, 0].axis("off")
        for col, (method_name, method_fn) in enumerate(methods.items(), start=1):
            denoised = method_fn(noisy)
            axes[row, col].imshow(denoised, cmap="gray")
            axes[row, col].set_title(method_name if row == 0 else "")
            axes[row, col].axis("off")

    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()
    print(f"[done] comparison figure saved -> {output_path}")


# ------------------------------------------------------------
# 3. RUN EVERYTHING (paste in a new cell, after your models are trained)
# ------------------------------------------------------------
print("this block ran successfully")

Saving everything to: /kaggle/working/paper_results_gaussian_only
this block ran successfully


In [9]:
model_dncnn = dncnn_model()
model_dncnn.summary()

model_resunet = resunet_model()
model_resunet.summary()

model_proposed = unet_model_4ch()
model_proposed.summary()

I0000 00:00:1782371771.837176      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782371771.843019      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "DnCNN"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 64, 64, 4) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 64, 64,    │      2,368 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 64, 64,    │     36,864 │ conv2d[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 64, 64,    │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 64,    │     36,864 │ activation[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 64, 64,    │     36,864 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 64, 64,    │     36,864 │ activation_2[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 64, 64,    │     36,864 │ activation_3[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_5[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 64, 64,    │          0 │ batch_normalizat

 Total params: 227,396 (888.27 KB)

 Trainable params: 226,628 (885.27 KB)

...

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 64, 64, 4) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_27 (Conv2D)  │ (None, 64, 64,    │      1,184 │ input_layer_2[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_28 (Conv2D)  │ (None, 64, 64,    │      9,248 │ conv2d_27[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 32, 32,    │          0 │ conv2d_28[0][0]   │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_29 (Conv2D)  │ (None, 32, 32,    │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_30 (Conv2D)  │ (None, 32, 32,    │     36,928 │ conv2d_29[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 16, 16,    │          0 │ conv2d_30[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_31 (Conv2D)  │ (None, 16, 16,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_32 (Conv2D)  │ (None, 16, 16,    │    147,584 │ conv2d_31[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 8, 8, 128) │          0 │ conv2d_32[0][0]   │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_33 (Conv2D)  │ (None, 8, 8, 256) │    295,168 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_34 (Conv2D)  │ (None, 8, 8, 256) │    590,080 │ conv2d_33[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose    │ (None, 16, 16,    │    131,200 │ conv2d_34[0][0]   │
│ (Conv2DTranspose)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 16, 16,    │          0 │ conv2d_transpose… │
│ (Concatenate)       │ 256)              │            │ conv2d_32[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_35 (Conv2D)  │ (None, 16, 16,    │    295,040 │ concatenate_2[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_36 (Conv2D)  │ (None, 16, 16,    │    147,584 │ conv2d_35[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_1  │ (None, 32, 32,    │     32,832 │ conv2d_36[0][0]   │
│ (Conv2DTranspose)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 1,925,988 (7.35 MB)

 Trainable params: 1,925,988 (7.35 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
# Step 1: Build dataset
dicom_paths = []
for root, _, files in os.walk('/kaggle/input/datasets/thepoojashinde/lidc-idri-denoising-subset/dataset/manifest-1600709154662/LIDC-IDRI/'):
    for f in files:
        if f.endswith('.dcm'):
            dicom_paths.append(os.path.join(root, f))
print(f"Found {len(dicom_paths)} DICOM files.")

X, Y, noise_levels = build_dwt_dataset_4ch(dicom_paths)
X_train, X_test, \
Y_train, Y_test, \
noise_train, noise_test = train_test_split(
    X,
    Y,
    noise_levels,
    test_size=0.2,
    random_state=42
)
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

# Step 2: Build + train all 3 models fresh
model_dncnn   = dncnn_model()
model_resunet = resunet_model()
model_proposed = unet_model_4ch(input_size=(64, 64, 4))

print("Starting DnCNN training...")
history_dncnn    = train_model(model_dncnn,    X_train, Y_train, "/kaggle/working/dncnn_dwt.keras", "DnCNN")

print("Starting ResUNet training...")
history_resunet  = train_model(model_resunet,  X_train, Y_train, "/kaggle/working/resunet_dwt.keras", "ResUNet")

print("Starting Proposed training...")
history_proposed = train_model(model_proposed, X_train, Y_train, "/kaggle/working/unet_dwt_v4_model.keras", "Proposed")

# Step 3: Save training curves
save_training_curve(history_dncnn, "DnCNN")
save_training_curve(history_resunet, "ResUNet")
save_training_curve(history_proposed, "Proposed")

print("ALL TRAINING DONE")

Found 3629 DICOM files.
Dataset created successfully.
Input shape : (3629, 64, 64, 4)
Target shape: (3629, 64, 64, 4)
X_train shape: (2903, 64, 64, 4), X_test shape: (726, 64, 64, 4)
Starting DnCNN training...
Epoch 1/50


I0000 00:00:1782371849.438431     150 service.cc:152] XLA service 0x78964000a470 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782371849.438469     150 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1782371849.438473     150 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1782371850.111681     150 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-06-25 07:17:31.840859: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-25 07:17:31.990948: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


  7/164 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - loss: 0.5663 - mae: 0.5666

I0000 00:00:1782371853.615751     150 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


163/164 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0856 - mae: 0.1453

...

164/164 ━━━━━━━━━━━━━━━━━━━━ 74s 255ms/step - loss: 0.3359 - mae: 0.1925 - val_loss: 0.0421 - val_mae: 0.1467
Epoch 2/50
164/164 ━━━━━━━━━━━━━━━━━━━━ 18s 111ms/step - loss: 0.0064 - mae: 0.0609 - val_loss: 0.0264 - val_mae: 0.0947
Epoch 3/50
164/164 ━━━━━━━━━━━━━━━━━━━━ 18s 112ms/step - loss: 0.0059 - mae: 0.0581 - val_loss: 0.0087 - val_mae: 0.0712
Epoch 4/50
164/164 ━━━━━━━━━━━━━━━━━━━━ 18s 113ms/step - loss: 0.0057 - mae: 0.0571 - val_loss: 0.0030 - val_mae: 0.0392
Epoch 5/50
164/164 ━━━━━━━━━━━━━━━━━━━━ 18s 109ms/step - loss: 0.0045 - mae: 0.0503 - val_loss: 0.0037 - val_mae: 0.0463
Epoch 6/50
164/164 ━━━━━━━━━━━━━━━━━━━━ 19s 114ms/step - loss: 0.0039 - mae: 0.0464 - val_loss: 0.0027 - val_mae: 0.0384
Epoch 7/50
164/164 ━━━━━━━━━━━━━━━━━━━━ 18s 110ms/step - loss: 0.0050 - mae: 0.0518 - val_loss: 0.0031 - val_mae: 0.0445
Epoch 8/50
164/164 ━━━━━━━━━━━━━━━━━━━━ 18s 111ms/step - loss: 0.0047 - mae: 0.0492 - val_loss: 0.0178 - val_mae: 0.0884
Epoch 9/50
164/164 ━━━━━━━━━━━━━━━━━━━━ 18s

2026-06-25 07:24:34.932042: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-25 07:24:35.072588: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-25 07:24:35.298918: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-25 07:24:35.442356: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


161/164 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0270 - mae: 0.0699

2026-06-25 07:24:44.022793: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-25 07:24:44.163234: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-25 07:24:44.304493: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


164/164 ━━━━━━━━━━━━━━━━━━━━ 23s 72ms/step - loss: 0.0078 - mae: 0.0368 - val_loss: 0.0012 - val_mae: 0.0206
Epoch 2/50
164/164 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - loss: 8.9100e-04 - mae: 0.0183 - val_loss: 8.2307e-04 - val_mae: 0.0181
Epoch 3/50
164/164 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - loss: 6.3088e-04 - mae: 0.0155 - val_loss: 6.0863e-04 - val_mae: 0.0151
Epoch 4/50
164/164 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - loss: 5.6844e-04 - mae: 0.0145 - val_loss: 5.3875e-04 - val_mae: 0.0139
Epoch 5/50
164/164 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - loss: 5.0598e-04 - mae: 0.0134 - val_loss: 5.0540e-04 - val_mae: 0.0133
Epoch 6/50
164/164 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - loss: 4.8573e-04 - mae: 0.0130 - val_loss: 4.9065e-04 - val_mae: 0.0130
Epoch 7/50
164/164 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - loss: 4.6180e-04 - mae: 0.0126 - val_loss: 4.7553e-04 - val_mae: 0.0128
Epoch 8/50
164/164 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - loss: 4.4694e-04 - mae: 0.0123 - val_loss: 4.5666e-04 - val_mae: 0.0124
Ep

In [12]:
models = {
    "DnCNN": model_dncnn,
    "ResUNet": model_resunet,
    "Proposed": model_proposed,
}
 
# Fair full test-set comparison (same n_samples for all 4, including BM3D)
testset_results = evaluate_full_testset(
    X_test,
    Y_test,
    noise_test,
    models
)
 
# Per-external-image tables (sir's Table I-VIII style) + visual grid
external_paths = [
    '/kaggle/input/datasets/thepoojashinde/lidc-idri-denoising-subset/test_dicom/lidc_subject_0001_slice_40.dcm',
    '/kaggle/input/datasets/thepoojashinde/lidc-idri-denoising-subset/test_dicom/lidc_subject_0023_slice_26.dcm',
    '/kaggle/input/datasets/thepoojashinde/lidc-idri-denoising-subset/test_dicom/1-4.dcm',
]
tables = run_external_image_tables(external_paths, models)
generate_comparison_figure(external_paths, models, std=0.07)
 
print("\\nAll files saved in:", FINAL_DIR)
import os
for f in sorted(os.listdir(FINAL_DIR)):
    print(" -", f)


Evaluating DnCNN on 726 samples...

Evaluating ResUNet on 726 samples...

Evaluating Proposed on 726 samples...

Evaluating BM3D on 726 samples (this will take a while)...
  BM3D progress: 0/726
  BM3D progress: 50/726
  BM3D progress: 100/726
  BM3D progress: 150/726
  BM3D progress: 200/726
  BM3D progress: 250/726
  BM3D progress: 300/726
  BM3D progress: 350/726
  BM3D progress: 400/726
  BM3D progress: 450/726
  BM3D progress: 500/726
  BM3D progress: 550/726
  BM3D progress: 600/726
  BM3D progress: 650/726
  BM3D progress: 700/726

FINAL TEST SET RESULTS (n=726, same for all methods)
  Method      MSE      PSNR     SSIM     FSIM  N_samples
   DnCNN 0.000490 33.601948 0.867983 0.788798        726
 ResUNet 0.002317 26.865843 0.609496 0.648710        726
Proposed 0.000247 36.652593 0.939452 0.878250        726
    BM3D 0.000431 34.813345 0.860085 0.814119        726
[done] Image1 tables saved
[done] Image2 tables saved
[done] Image3 tables saved
[done] comparison figure saved -> /